# 03 — Generative AI Concepts, With LangChain
Same tasks as notebook 02, now via LangChain's abstractions. Compare the two side by side — that comparison *is* the learning goal here: see exactly what the framework buys you (reusability, parsing, composability) versus what it costs (another layer to understand when something breaks).

# Setup
Run this first in every notebook. It assumes this notebook lives in the same
folder as `inhouse_wrappers.py`, `rag_pure_python.py`, and `inhouse_llm.py`
(the files from earlier in this project). If not, add the folder to `sys.path`.

In [ ]:
import sys, os
# sys.path.append("/path/to/inhouse_rag_capstone")  # uncomment & adjust if needed

from inhouse_llm import (
    multimodal_chat, get_embedding,
    MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL,
    MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B, MODEL_JINA,
)
from inhouse_wrappers import InHouseLLM, InHouseEmbeddings, llm_for
from rag_pure_python import chunk_text, SimpleVectorStore, generate_answer

print("Setup OK")

In [ ]:
# pip install langchain langchain-core --break-system-packages
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

## 1. PromptTemplate — parameterized prompts
**Why:** instead of f-string concatenation, templates make prompts reusable, testable, and swappable. **When:** any prompt you'll reuse with different inputs more than once — which in practice is almost every prompt in a real app.

In [ ]:
template = PromptTemplate.from_template(
    "Classify sentiment as positive, negative, or neutral. Reply with one word.\n\nText: {text}"
)
llm = InHouseLLM(model=MODEL_QWEN3_14B, max_tokens=50)

chain = template | llm | StrOutputParser()
result = chain.invoke({"text": "The model latency improved a lot after the upgrade."})
print(result)

## 2. LCEL chains — composing steps with `|`
**Why:** the pipe operator (LangChain Expression Language) lets you compose prompt -> model -> parser -> next step as one declarative pipeline instead of manual function calls. Easier to swap pieces (e.g., change the parser) without touching the rest.

In [ ]:
cot_template = ChatPromptTemplate.from_messages([
    ("system", "Think step by step, then give the final answer on the last line as 'Answer: <n>'."),
    ("user", "{question}")
])

cot_chain = cot_template | llm | StrOutputParser()
print(cot_chain.invoke({"question":
    "A retriever returns 5 chunks. 2 are irrelevant. Of the relevant ones, "
    "half mention MCP. How many chunks mention MCP?"}))

## 3. Structured output with `JsonOutputParser`
**Why:** instead of manually `json.loads()`-ing and hoping, LangChain's parser can retry/validate. This is the framework-provided version of notebook 02 section 3.

In [ ]:
json_template = ChatPromptTemplate.from_messages([
    ("system", "Respond with ONLY valid JSON, no markdown fences. "
               "Schema: {{\"model\": str, \"use_case\": str, \"confidence\": float}}"),
    ("user", "{task}")
])

json_chain = json_template | llm | JsonOutputParser()
result = json_chain.invoke({"task": "Recommend which in-house model fits 'summarizing a 50-page PDF'."})
print(result, type(result))

## 4. Memory — multi-turn conversation
**Why:** raw API calls are stateless; a chat app needs history. LangChain's message history utilities handle the bookkeeping. **When:** any multi-turn interaction — most agent and chatbot use cases.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

history = [SystemMessage(content="You are a helpful assistant.")]

def chat_turn(user_text):
    history.append(HumanMessage(content=user_text))
    # InHouseLLM is a plain LLM (string in/out), so flatten messages to one prompt string
    flat_prompt = "\n".join(
        f"{m.type.upper()}: {m.content}" for m in history
    )
    reply = llm.invoke(flat_prompt)
    history.append(AIMessage(content=reply))
    return reply

print(chat_turn("What's RAG?"))
print(chat_turn("Now explain it to a 10-year-old."))  # relies on prior turn's context

### Side-by-side takeaway
Re-read 02 and 03 back to back. Notice: LangChain didn't make the *model* smarter — it made the *plumbing* (templating, parsing, history, composition) more reusable. For a one-off script, raw calls (02) are simpler. For an app with many prompts reused across features, LangChain's structure pays for itself.